# 03 - Modelagem com indicadores World Bank

Este notebook avalia se indicadores socioeconômicos do World Bank melhoram o desempenho dos modelos em relação ao uso apenas de features históricas e temporais de conflito.

A comparação é feita no mesmo conjunto de países/anos do dataset integrado para evitar comparação injusta entre amostras diferentes.

In [1]:
from pathlib import Path

import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer


current_path = Path.cwd()

if (current_path / "data").exists():
    PROJECT_ROOT = current_path
else:
    PROJECT_ROOT = current_path.parents[1]

DATA_PATH = PROJECT_ROOT / "data" / "final" / "conflict_country_year_world_bank.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
print("Countries:", df["country"].nunique())
print("Years:", df["year"].min(), "-", df["year"].max())
df.head()

Dataset loaded successfully.
Shape: (6663, 39)
Countries: 195
Years: 1989 - 2023


,country,year,region,main_government_name,state_based_conflict_exists,state_based_dyad_count,state_based_deaths_best,intrastate_conflict_exists,intrastate_deaths_best,interstate_conflict_exists,...,world_bank_country_name,review_status,world_bank_region,world_bank_income_level,population_total,gdp_per_capita_current_usd,gdp_growth_annual_pct,inflation_consumer_prices_annual_pct,unemployment_total_pct,military_expenditure_pct_gdp
0,United States of America,1989,Americas,Government of United States of America,0,0,0,0,0,0,...,United States,manual_review,North America,High income,246819000.0,22857.154433,3.672238,4.827003,NaN,5.871206
1,United States of America,1990,Americas,Government of United States of America,0,0,0,0,0,0,...,United States,manual_review,North America,High income,249623000.0,23888.600009,1.885966,5.397956,NaN,5.605175
2,United States of America,1991,Americas,Government of United States of America,0,0,0,0,0,0,...,United States,manual_review,North America,High income,252981000.0,24342.258905,-0.108313,4.234964,6.8,4.883429
3,United States of America,1992,Americas,Government of United States of America,0,0,0,0,0,0,...,United States,manual_review,North America,High income,256514000.0,25418.990776,3.522497,3.028820,7.5,4.970467
4,United States of America,1993,Americas,Government of United States of America,0,0,0,0,0,0,...,United States,manual_review,North America,High income,259919000.0,26387.293734,2.751796,2.951657,6.9,4.604350


In [2]:
TARGET_COLUMN = "target_conflict_next_year"
TRAIN_END_YEAR = 2016

BASE_FEATURE_COLUMNS = [
    "year",
    "state_based_conflict_exists",
    "state_based_dyad_count",
    "state_based_deaths_best",
    "intrastate_conflict_exists",
    "intrastate_deaths_best",
    "interstate_conflict_exists",
    "interstate_deaths_best",
    "non_state_conflict_exists",
    "non_state_dyad_count",
    "non_state_deaths_best",
    "one_sided_violence_exists",
    "one_sided_dyad_count",
    "one_sided_deaths_best",
    "cumulative_organized_violence_deaths_best",
    "organized_violence_exists",
]

TEMPORAL_FEATURE_COLUMNS = [
    "conflict_previous_year",
    "conflict_last_3_years_count",
    "conflict_last_5_years_count",
    "deaths_previous_year",
    "deaths_last_3_years_sum",
    "deaths_last_5_years_sum",
    "years_since_last_conflict",
]

WORLD_BANK_FEATURE_COLUMNS = [
    "population_total",
    "gdp_per_capita_current_usd",
    "gdp_growth_annual_pct",
    "inflation_consumer_prices_annual_pct",
    "unemployment_total_pct",
    "military_expenditure_pct_gdp",
]

EXPERIMENTS = {
    "temporal_only": BASE_FEATURE_COLUMNS + TEMPORAL_FEATURE_COLUMNS,
    "temporal_world_bank": BASE_FEATURE_COLUMNS + TEMPORAL_FEATURE_COLUMNS + WORLD_BANK_FEATURE_COLUMNS,
}

print("Target distribution:")
print(df[TARGET_COLUMN].value_counts(normalize=True).sort_index().round(4))

print("\nMissing values in World Bank features:")
print(df[WORLD_BANK_FEATURE_COLUMNS].isna().mean().mul(100).round(2).sort_values(ascending=False))

Target distribution:
target_conflict_next_year
0    0.7018
1    0.2982
Name: proportion, dtype: float64

Missing values in World Bank features:
military_expenditure_pct_gdp            24.19
inflation_consumer_prices_annual_pct    13.18
unemployment_total_pct                  12.71
gdp_growth_annual_pct                    2.06
gdp_per_capita_current_usd               1.56
population_total                         0.00
dtype: float64


In [3]:
train_mask = df["year"] <= TRAIN_END_YEAR
test_mask = df["year"] > TRAIN_END_YEAR

print("Train period:", df.loc[train_mask, "year"].min(), "-", df.loc[train_mask, "year"].max())
print("Test period:", df.loc[test_mask, "year"].min(), "-", df.loc[test_mask, "year"].max())

print("\nTrain rows:", train_mask.sum())
print("Test rows:", test_mask.sum())

print("\nTrain target distribution:")
print(df.loc[train_mask, TARGET_COLUMN].value_counts(normalize=True).sort_index().round(4))

print("\nTest target distribution:")
print(df.loc[test_mask, TARGET_COLUMN].value_counts(normalize=True).sort_index().round(4))

Train period: 1989 - 2016
Test period: 2017 - 2023

Train rows: 5305
Test rows: 1358

Train target distribution:
target_conflict_next_year
0    0.7086
1    0.2914
Name: proportion, dtype: float64

Test target distribution:
target_conflict_next_year
0    0.6753
1    0.3247
Name: proportion, dtype: float64


In [4]:
def evaluate_predictions(experiment_name, model_name, y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        "experiment": experiment_name,
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }


def build_models():
    return {
        "Logistic Regression scaled": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=5000,
                class_weight="balanced",
                random_state=42,
            )),
        ]),
        "Decision Tree": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", DecisionTreeClassifier(
                max_depth=4,
                class_weight="balanced",
                random_state=42,
            )),
        ]),
        "Random Forest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                n_estimators=300,
                max_depth=6,
                min_samples_leaf=5,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1,
            )),
        ]),
    }

In [5]:
results = []

y_test = df.loc[test_mask, TARGET_COLUMN]
y_pred_persistence = df.loc[test_mask, "organized_violence_exists"]

results.append(
    evaluate_predictions(
        "reference",
        "Persistence baseline",
        y_test,
        y_pred_persistence,
    )
)

for experiment_name, feature_columns in EXPERIMENTS.items():
    X_train = df.loc[train_mask, feature_columns]
    y_train = df.loc[train_mask, TARGET_COLUMN]

    X_test = df.loc[test_mask, feature_columns]
    y_test = df.loc[test_mask, TARGET_COLUMN]

    models = build_models()

    for model_name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        results.append(
            evaluate_predictions(
                experiment_name,
                model_name,
                y_test,
                y_pred,
            )
        )

results_df = pd.DataFrame(results)
results_df.round(4)

,experiment,model,accuracy,precision,recall,f1_score,tn,fp,fn,tp
0,reference,Persistence baseline,0.9072,0.8571,0.8571,0.8571,854,63,63,378
1,temporal_only,Logistic Regression scaled,0.9161,0.8959,0.8390,0.8665,874,43,71,370
2,temporal_only,Decision Tree,0.8895,0.8198,0.8458,0.8326,835,82,68,373
3,temporal_only,Random Forest,0.9006,0.8312,0.8707,0.8505,839,78,57,384
4,temporal_world_bank,Logistic Regression scaled,0.9153,0.8918,0.8413,0.8658,872,45,70,371
5,temporal_world_bank,Decision Tree,0.9013,0.8218,0.8889,0.8540,832,85,49,392
6,temporal_world_bank,Random Forest,0.9043,0.8344,0.8798,0.8565,840,77,53,388


In [6]:
results_sorted = results_df.sort_values(
    by=["f1_score", "recall", "precision"],
    ascending=False,
)

results_sorted.round(4)

,experiment,model,accuracy,precision,recall,f1_score,tn,fp,fn,tp
1,temporal_only,Logistic Regression scaled,0.9161,0.8959,0.8390,0.8665,874,43,71,370
4,temporal_world_bank,Logistic Regression scaled,0.9153,0.8918,0.8413,0.8658,872,45,70,371
0,reference,Persistence baseline,0.9072,0.8571,0.8571,0.8571,854,63,63,378
6,temporal_world_bank,Random Forest,0.9043,0.8344,0.8798,0.8565,840,77,53,388
5,temporal_world_bank,Decision Tree,0.9013,0.8218,0.8889,0.8540,832,85,49,392
3,temporal_only,Random Forest,0.9006,0.8312,0.8707,0.8505,839,78,57,384
2,temporal_only,Decision Tree,0.8895,0.8198,0.8458,0.8326,835,82,68,373


In [7]:
baseline_f1 = results_df.loc[
    results_df["model"] == "Persistence baseline",
    "f1_score"
].iloc[0]

results_with_diff = results_df.copy()
results_with_diff["f1_difference_vs_persistence"] = (
    results_with_diff["f1_score"] - baseline_f1
)

results_with_diff.sort_values("f1_score", ascending=False).round(4)

,experiment,model,accuracy,precision,recall,f1_score,tn,fp,fn,tp,f1_difference_vs_persistence
1,temporal_only,Logistic Regression scaled,0.9161,0.8959,0.8390,0.8665,874,43,71,370,0.0094
4,temporal_world_bank,Logistic Regression scaled,0.9153,0.8918,0.8413,0.8658,872,45,70,371,0.0087
0,reference,Persistence baseline,0.9072,0.8571,0.8571,0.8571,854,63,63,378,0.0000
6,temporal_world_bank,Random Forest,0.9043,0.8344,0.8798,0.8565,840,77,53,388,-0.0006
5,temporal_world_bank,Decision Tree,0.9013,0.8218,0.8889,0.8540,832,85,49,392,-0.0031
3,temporal_only,Random Forest,0.9006,0.8312,0.8707,0.8505,839,78,57,384,-0.0066
2,temporal_only,Decision Tree,0.8895,0.8198,0.8458,0.8326,835,82,68,373,-0.0246


In [8]:
OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)

results_path = OUTPUT_TABLES_DIR / "world_bank_feature_model_results.csv"

results_with_diff.to_csv(results_path, index=False)

print(f"Saved results to: {results_path}")

Saved results to: C:\Users\enzo.going\Documents\GitHub\international-conflict-risk-ml\outputs\tables\world_bank_feature_model_results.csv


## Interpretação esperada

A pergunta principal deste notebook é:

> Indicadores socioeconômicos do World Bank melhoram a previsão em relação ao uso apenas do histórico temporal de conflito?

A comparação deve considerar a baseline de persistência como referência obrigatória.